In [2]:
# 导入所有必需的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops
import math

# 10. LoRA Tutorial | 参数高效微调: 深入剖析 LoRA (PEFT)

###  核心思想与痛点

> **为什么需要 LoRA？**
> 全参微调 (Full Fine-tuning) 一个 7B 模型需要大规模的显存来保存优化器状态（Adam 需要保存参数的动量和方差，占用额外 8 倍参数量的显存）。绝大多数中小企业和个人开发者无法承担。
> **LoRA 的本质：**
> 冻结原始的预训练模型权重，并在每个 Dense 层旁边注入可训练的“旁路”降秩矩阵（A 和 B）。微调时只更新这非常少量的参数。最终推理时，可以将旁路权重无损“合并（Merge）”回主权重中。


###  核心公式与张量维度

**前向传播公式：**
给定预训练权重 $W_0 \in \mathbb{R}^{d \times k}$，输入 $x$，LoRA 修改后的输出为：
$$ h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} B A x $$

*   $A \in \mathbb{R}^{r \times k}$：降维矩阵，通常使用随机高斯分布初始化（Kaiming Uniform）。
*   $B \in \mathbb{R}^{d \times r}$：升维矩阵，**必须初始化为全 0**，以保证初始状态下 $\Delta W = 0$，也就是微调前的输出和预训练模型完全一致。
*   $r$ (rank)：矩阵的秩，通常设置极小，如 8 或 16。
*   $\alpha$：缩放因子（Scaling Factor），用来控制 $\Delta W$ 的影响程度。

**推理时合并权重 (Merge Weights)：**
$$ W_{\text{merged}} = W_0 + \frac{\alpha}{r} B A $$
这样在部署时，计算图里没有 A 和 B，完全没有额外的推理耗时（No Inference Latency）。


### 1. 为什么多了 $A$ 和 $B$ 反而节省显存？

显存占用的核心瓶颈并非“存参数”，而是**存梯度（Gradient）**和**存优化器状态（Optimizer States）**。

#### 训练阶段（显存节省的核心）
在训练过程中，原始权重 $W_0$ 被完全冻结。这意味着系统无需为 $W_0$ 计算梯度，更无需为其存储优化器状态（如 Adam 的动量与方差）。系统只需为新增的小矩阵 $A$ 和 $B$ 分配梯度与状态空间。

| 对比项 | 全量微调 (Full Fine-tuning) | LoRA 微调 |
| :--- | :--- | :--- |
| **原始权重 $W_0$ 状态** | 参与反向传播，需存梯度和优化器状态 | **冻结**，仅作只读存储，无需梯度和状态 |
| **新增参数数量级** | 无 | $d \times r + r \times k$ |
| **显存占用公式** | $3 \times (d \times k)$ | $3 \times (d \times r + r \times k)$ （$W_0$ 仅占 1 份只读） |

因为 $r \ll \min(d, k)$，新增参数量远小于原始矩阵，因此**大幅削减了梯度与优化器带来的显存开销**。

#### 推理阶段（零额外开销）
推理部署前，执行合并操作：
$$ W_{\text{merged}} = W_0 + \frac{\alpha}{r} B A $$
合并后，$A$ 和 $B$ 矩阵被彻底丢弃。推理计算图中仅存在 $W_{\text{merged}}$，因此**推理显存占用和计算速度与原始预训练模型完全一致**，没有任何额外负担。

---

### 2. 线性代数基石：秩（Rank）到底是什么？

#### 数学定义
矩阵的秩等于该矩阵中**线性无关的行（或列）的最大数目**。它衡量了矩阵能够覆盖的“有效空间维度”或“独立变化方向的个数”。

#### “信息管道”比喻
- 若 $A$ 的形状为 $r \times k$（降维），其秩 **最多为 $r$**（行数上限）。
- 若 $B$ 的形状为 $d \times r$（升维），其秩 **最多为 $r$**（列数上限）。
- 矩阵乘积的秩遵循铁律：$\text{秩}(B \times A) \le \min(\text{秩}(B), \text{秩}(A))$。

因此，$\text{秩}(B \times A) \le r$。直观理解：输入 $x$ 从 $k$ 维被 $A$ 强行压缩进仅有 $r$ 条车道的“隧道”，$B$ 再将其扩展回 $d$ 维。无论最终输出多么庞大，其**内在的独立变化自由度**被锁死在了 $r$ 这个数量级上，这就是“低秩”的物理含义。

---

### 3. 为什么要强行使用低秩（Low-Rank）？

这是基于预训练模型**“内在维度（Intrinsic Dimension）”**的深刻洞察。

- **高秩（全量微调）**：允许在 $d \times k$ 的完整空间内调整每一个参数。这虽然灵活，但极易导致过拟合，且计算成本极高。
- **低秩（LoRA）**：大量实验（如 Aghajanyan 等的研究）证明，预训练模型适应下游任务时，其所需的**权重增量（$\Delta W$）** 本质上是“低秩”的。也就是说，虽然 $\Delta W$ 矩阵尺寸巨大，但它实际躺在一个极其低维的子空间中。

LoRA 强行用 $BA$ 来建模 $\Delta W$，在数学上恰好契合了微调过程的本质——只需要调整少数几个关键的“特征方向”即可适应新任务，同时顺带获得了显存与速度的极大收益。

---

### 4. LoRA 前向传播公式的逐项拆解

给定预训练权重 $W_0 \in \mathbb{R}^{d \times k}$，输入 $x$，LoRA 修改后的输出为：
$$ h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} B A x $$

我们将信息流分为两条路径：

- **$W_0 x$（主干道，旧知识）**  
  输入 $x$ 经过冻结的原始权重，保留模型原本的通用知识与能力。该路径恒定不变，确保模型不会发生“灾难性遗忘”。

- **$A x$（新路前半段，特征筛选）**  
  $A$ 的形状为 $r \times k$。它将高维输入 $x$ 压缩到极低的 $r$ 维空间。作用类似于“信息筛子”，只提取出与当前新任务最相关的核心特征。

- **$B (A x)$（新路后半段，增量生成）**  
  $B$ 的形状为 $d \times r$。它将压缩后的 $r$ 维特征映射回 $d$ 维空间，生成对原始输出的修正量 $\Delta W x$。

- **$\frac{\alpha}{r}$（缩放因子，控制步长）**  
  该系数用于控制新路的影响力。由于 $B$ 和 $A$ 的乘积范数大致与 $r$ 成正比，**除以 $r$** 可以保证无论设置 $r=4$ 还是 $r=64$，新路输出的初始量级保持稳定，不影响预训练输出的量级。超参数 $\alpha$ 则作为“新任务学习率倍数”，$\alpha$ 越大，模型越倾向于适应新数据。

---

**关于初始化的补充：**  
矩阵 $A$ 通常使用随机高斯分布或 Kaiming Uniform 初始化，而矩阵 $B$ **必须初始化为全零矩阵**。这使得训练开始时 $BA = \mathbf{0}$，从而保证 $\Delta W = 0$，即微调前的输出与原始预训练模型完全一致，训练起点非常稳定。

In [1]:
class LoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, r: int = 8, lora_alpha: int = 16):
        super().__init__()
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.r
        
        # ==========================================
        # TODO 1: 初始化主权重和 LoRA 矩阵
        # ==========================================
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.linear.weight.requires_grad = False
        
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, r))
        
        self.reset_parameters()   

    def reset_parameters(self):
        # ==========================================
        # TODO 2: 初始化权重
        # ==========================================
        nn.init.kaiming_uniform_(self.linear.weight, a=math.sqrt(5))
        
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 3: 实现前向传播
        # 1. 计算主权重的输出
        # 2. 计算 LoRA 分支的输出（先降维再升维，最后乘以缩放因子）
        # 3. 将两者相加
        # 提示: 注意矩阵转置和乘法顺序
        # ==========================================
        result = self.linear(x)
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T * self.scaling
        result += lora_out
        return result

    def merge_weights(self):
        # ==========================================
        # TODO 4: 合并权重（零延迟推理）
        # 提示: 将 LoRA 的低秩更新合并到主权重中
        # ==========================================
        self.linear.weight.data += (self.lora_B @ self.lora_A) * self.scaling
        